# LeetCode 4018 — Total Sum of Interaction Cost in Tree Groups II bằng Python

- Link: https://leetcode.com/problems/total-sum-of-interaction-cost-in-tree-groups-ii/
- Độ khó: **Hard**
- Chủ đề: Cây, Đóng góp của từng cạnh (edge contribution), Gộp nhỏ vào lớn (small-to-large), Cây ảo (virtual tree) + LCA

> **Bài Premium:** đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium (đề thực tế lấy từ bản doocs). Tên phương thức dưới đây (`interactionCosts`) là **suy luận** từ bài cùng họ "Total Sum of Interaction Cost in Tree Groups"; nếu LeetCode hiển thị tên khác, chỉ cần đổi tên phương thức, phần thân không đổi.

Skeleton (Python3):

```python
class Solution:
    def interactionCosts(self, n: int, edges: list[list[int]], group: list[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho một cây `n` nút (gốc 0) và nhãn nhóm `group[i]`. Chi phí tương tác giữa hai nút là **khoảng cách** (số cạnh) trên cây. Tính **tổng chi phí của mọi cặp** `(u, v)`, `u < v`, thuộc **cùng nhóm**. Với `n` tới `10^5` và nhãn nhóm tới `n`, không thể xét từng cặp.

Chìa khoá: **đóng góp theo cạnh**. Trong cây, đường đi giữa hai nút là duy nhất, nên mỗi cạnh được dùng bởi đúng những cặp nằm ở hai phía khác nhau của cạnh. Do đó mỗi cạnh đóng góp `Σ_nhóm (số nút của nhóm ở phía dưới) × (số nút của nhóm ở phía trên)`. Khó khăn thực sự: **quản lý đếm theo nhóm cho mọi cây con** — dùng kỹ thuật **gộp nhỏ vào lớn**.

Sau notebook này bạn sẽ:

1. Biểu diễn cây bằng danh sách kề và duyệt **không đệ quy** (BFS) để tránh tràn ngăn xếp.
2. Hiểu và chứng minh công thức "đóng góp của một cạnh" `Σ_g c_g (T_g − c_g)`.
3. Dùng `dict` đếm theo nhóm trong từng cây con, và **gộp nhỏ vào lớn** để tổng chi phí gộp là `O(n log n)`.
4. Duy trì tổng `Σ c_g (T_g − c_g)` **tăng dần** khi gộp (không tính lại từ đầu).
5. Biết một cách khác: **cây ảo theo từng nhóm** + LCA; và đối chiếu mọi cách bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Cây vô hướng `n` nút (`0..n-1`), gốc 0, cho bởi `edges` (`n − 1` cạnh).
- `group[i]` là nhãn nhóm của nút `i`. Hai nút cùng nhóm ⇔ nhãn bằng nhau.
- Chi phí tương tác giữa hai nút = số cạnh trên đường đi duy nhất giữa chúng.
- Trả về **tổng chi phí** trên mọi cặp `(u, v)` với `0 <= u < v < n` và `group[u] == group[v]`.

**Ví dụ 1:** `n = 3`, `edges = [[0,1],[1,2]]`, `group = [1,1,1]` → `4` (`1 + 1 + 2`).
**Ví dụ 2:** `n = 3`, `edges = [[0,1],[1,2]]`, `group = [3,2,3]` → `2` (chỉ cặp `(0, 2)`).
**Ví dụ 3:** `n = 4`, `edges = [[0,1],[0,2],[0,3]]`, `group = [1,1,4,4]` → `3` (nhóm 1: `(0,1)` = 1; nhóm 4: `(2,3)` = 2).
**Ví dụ 4:** `n = 2`, `edges = [[0,1]]`, `group = [1,2]` → `0`.

**Ràng buộc**

- `1 <= n <= 10^5`, `edges.length == n - 1`, `0 <= u_i, v_i <= n - 1`
- `group.length == n`, `1 <= group[i] <= n`
- `edges` là một cây hợp lệ.

In [1]:
from collections import Counter, deque
import random

EXAMPLES = [
    (3, [[0, 1], [1, 2]], [1, 1, 1], 4),
    (3, [[0, 1], [1, 2]], [3, 2, 3], 2),
    (4, [[0, 1], [0, 2], [0, 3]], [1, 1, 4, 4], 3),
    (2, [[0, 1]], [1, 2], 0),
]
for n, edges, group, e in EXAMPLES:
    print(f"n={n}, edges={edges}, group={group} -> kỳ vọng {e}")

n=3, edges=[[0, 1], [1, 2]], group=[1, 1, 1] -> kỳ vọng 4
n=3, edges=[[0, 1], [1, 2]], group=[3, 2, 3] -> kỳ vọng 2
n=4, edges=[[0, 1], [0, 2], [0, 3]], group=[1, 1, 4, 4] -> kỳ vọng 3
n=2, edges=[[0, 1]], group=[1, 2] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Danh sách kề (adjacency list)

Một cây/đồ thị lưu bằng `adj[u]` = danh sách các nút kề `u`. Với cạnh vô hướng `[u, v]` ta thêm `v` vào `adj[u]` **và** `u` vào `adj[v]`.

In [2]:
n = 4
edges = [[0, 1], [0, 2], [0, 3]]
adj = [[] for _ in range(n)]
for u, v in edges:
    adj[u].append(v)
    adj[v].append(u)
print(adj)

[[1, 2, 3], [0], [0], [0]]


### 3.2. Duyệt cây không đệ quy: BFS, mảng `parent` và `order`

Đệ quy trên cây dạng đường thẳng `10^5` nút vượt giới hạn đệ quy của Python (mặc định ~1000). Ta thay bằng BFS: bắt đầu từ gốc 0, ghi `parent[w] = v` khi gặp nút mới `w` từ `v`, và lưu `order` = thứ tự duyệt. Tính chất quan trọng: **cha luôn đứng trước con trong `order`**, nên duyệt `order` **ngược** là xử lý con trước cha (từ lá lên gốc).

In [3]:
def bfs_order(n, adj, root=0):
    parent = [-1] * n
    order = [root]
    seen = [False] * n
    seen[root] = True
    for v in order:                      # order vừa duyệt vừa dài thêm
        for w in adj[v]:
            if not seen[w]:
                seen[w] = True
                parent[w] = v
                order.append(w)
    return parent, order

edges = [[0, 1], [1, 2], [1, 3], [0, 4]]
n = 5
adj = [[] for _ in range(n)]
for u, v in edges:
    adj[u].append(v); adj[v].append(u)
parent, order = bfs_order(n, adj)
print("parent =", parent)
print("order  =", order, "-> ngược:", order[::-1])

parent = [-1, 0, 1, 1, 0]
order  = [0, 1, 4, 2, 3] -> ngược: [3, 2, 4, 1, 0]


### 3.3. Đếm theo nhóm: `dict` và `Counter`

`Counter(group)` cho `T[g]` = số nút của nhóm `g` toàn cây. Một `dict` `cnt` cho cây con: `cnt[g]` = số nút nhóm `g` trong cây con đó. `cnt.get(g, 0)` đọc an toàn khi `g` chưa có.

In [4]:
group = [1, 1, 4, 4, 1]
T = Counter(group)
print(T, T[1], T[4], T[9])          # T[9] = 0 (Counter trả 0 cho khoá vắng)
cnt = {}
for g in [1, 4, 1]:
    cnt[g] = cnt.get(g, 0) + 1
print(cnt)

Counter({1: 3, 4: 2}) 3 2 0
{1: 2, 4: 1}


### 3.4. Mỗi cạnh chia cây thành hai phần

Cắt cạnh `(v, parent[v])` thì cây tách thành **cây con của `v`** và **phần còn lại**. Hai nút `x, y` có đường đi qua cạnh đó ⇔ chúng nằm ở **hai phía khác nhau**. Do đường đi trong cây duy nhất:

`dist(x, y) = số cạnh mà x và y nằm ở hai phía khác nhau`.

Cộng trên mọi cặp cùng nhóm ⇒ tổng `= Σ_cạnh (số cặp cùng nhóm bị cạnh đó tách)`.

In [5]:
# Ví dụ: đường thẳng 0-1-2-3. Cạnh (1,2) tách {0,1} | {2,3}: các cặp bị tách: (0,2),(0,3),(1,2),(1,3)
n = 4
sub = {0, 1}
cross = [(x, y) for x in range(n) for y in range(x + 1, n) if (x in sub) != (y in sub)]
print(cross, len(cross))
# Tổng khoảng cách mọi cặp = Σ_cạnh (kích thước phía dưới × kích thước phía trên):  3*1 + 2*2 + 1*3 = 10
print(3 * 1 + 2 * 2 + 1 * 3, "và", sum(abs(x - y) for x in range(n) for y in range(x + 1, n)))

[(0, 2), (0, 3), (1, 2), (1, 3)] 4
10 và 10


### 3.5. Gộp nhỏ vào lớn (small-to-large)

Khi gộp hai `dict`, nếu luôn **đổ cái nhỏ vào cái lớn** thì mỗi phần tử bị di chuyển nhiều nhất `log2(n)` lần (mỗi lần di chuyển kích thước tập chứa nó **ít nhất gấp đôi**). Tổng công sức `O(n log n)`. Trong Python, ta hoán đổi tham chiếu: `if len(a) < len(b): a, b = b, a`.

In [6]:
a = {1: 5, 2: 1}
b = {1: 2, 3: 7, 4: 1}
moved = 0
if len(a) < len(b):
    a, b = b, a                      # a luôn là dict lớn hơn
for k, v in b.items():
    a[k] = a.get(k, 0) + v
    moved += 1
print(a, "| số phần tử phải di chuyển:", moved)

{1: 7, 3: 7, 4: 1, 2: 1} | số phần tử phải di chuyển: 2


## 4. Ý tưởng nền tảng / trực giác

**Tính tay ví dụ 3** (sao 0–{1,2,3}, `group = [1,1,4,4]`). Tổng số nút mỗi nhóm: `T[1] = 2`, `T[4] = 2`.

| Cạnh | Cây con phía dưới | Nhóm trong cây con | Đóng góp `Σ c(T − c)` |
|---|---|---|---|
| (1, 0) | `{1}` | nhóm 1: `c = 1` | `1·(2 − 1) = 1` |
| (2, 0) | `{2}` | nhóm 4: `c = 1` | `1·(2 − 1) = 1` |
| (3, 0) | `{3}` | nhóm 4: `c = 1` | `1·(2 − 1) = 1` |

Tổng `1 + 1 + 1 = 3`. Khớp đáp án. Cặp `(2, 3)` nhóm 4 có khoảng cách 2 được tính qua **hai** cạnh `(2,0)` và `(3,0)` — mỗi cạnh một đơn vị.

**Ví dụ 2** (đường `0–1–2`, `group = [3,2,3]`, `T[3] = 2`): cạnh `(2,1)`: cây con `{2}` có nhóm 3 `c = 1` ⇒ 1; cạnh `(1,0)`: cây con `{1,2}`: nhóm 3 `c = 1` (nút 2), nhóm 2 `c = 1` nhưng `T[2] = 1` ⇒ `c(T − c) = 0` ⇒ 1. Tổng 2. ✔

## 5. Cách 1 — Vét cạn: BFS từ mỗi nút, cộng khoảng cách các cặp cùng nhóm

**Ý tưởng.** Với mỗi nút `s`, BFS tính khoảng cách tới mọi nút; cộng `dist(s, t)` cho các `t > s` cùng nhóm. Đây là nguyên văn đề bài, ta dùng làm trọng tài.

In [7]:
def total_cost_bruteforce(n, edges, group):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    total = 0
    for s in range(n):
        dist = [-1] * n
        dist[s] = 0
        queue = deque([s])
        while queue:
            x = queue.popleft()
            for y in adj[x]:
                if dist[y] < 0:
                    dist[y] = dist[x] + 1
                    queue.append(y)
        for t in range(s + 1, n):
            if group[s] == group[t]:
                total += dist[t]
    return total

for n, edges, group, e in EXAMPLES:
    print(total_cost_bruteforce(n, edges, group), e)

4 4
2 2
3 3
0 0


**Độ phức tạp.** `n` lần BFS, mỗi lần `O(n)` ⇒ `O(n²)`. Với `n = 10^5` là `10^10` — quá chậm.

**Nút thắt:** ta tính lại các khoảng cách chung rất nhiều lần; mỗi cạnh được "đi qua" bởi nhiều cặp. Hãy đếm theo **cạnh** thay vì theo **cặp**.

## 6. Bước đột phá

### 6.1. Đóng góp của một cạnh

Gọi `T[g]` là số nút nhóm `g` toàn cây, và `c_g(v)` là số nút nhóm `g` trong **cây con của `v`** (gốc 0). Cạnh `(v, parent[v])` tách `c_g(v)` nút nhóm `g` ở dưới khỏi `T[g] − c_g(v)` nút nhóm `g` ở trên. Số cặp cùng nhóm bị cạnh này tách là

`E(v) = Σ_g c_g(v) · (T[g] − c_g(v))`.

Đáp án `= Σ_{v ≠ 0} E(v)`.

### 6.2. Viết lại `E(v)` để cập nhật tăng dần

`E(v) = Σ_g c_g(v)·T[g] − Σ_g c_g(v)² = A(v) − B(v)`, với `A(v) = Σ_g c_g·T[g]` và `B(v) = Σ_g c_g²`. Khi một nhóm `g` có số đếm tăng từ `y` lên `y + x`:

- `A` tăng `x · T[g]`.
- `B` tăng `(y + x)² − y²`.

Như vậy khi **gộp** cây con con vào cây con cha, ta chỉ cần xử lý từng cặp `(g, x)` của dict **nhỏ** và cập nhật `A, B` — không cần quét toàn bộ dict lớn.

### 6.3. Gộp nhỏ vào lớn

Mỗi nút có một cấu trúc `(cnt, A, B)`; khởi tạo `cnt = {group[v]: 1}`, `A = T[group[v]]`, `B = 1`. Xử lý các nút theo thứ tự BFS **ngược** (con trước cha): khi tới `v`, mọi con của nó đã được gộp vào `v`, nên `E(v) = A − B` đã sẵn sàng; cộng vào đáp án (nếu `v ≠ 0`), rồi **gộp** cấu trúc của `v` vào cha `p`: nếu dict của `v` lớn hơn dict của `p` thì **hoán đổi** cả bộ `(cnt, A, B)`, sau đó duyệt dict nhỏ để cập nhật.

**Quy tắc nếu… thì…:**
- Nếu một nhóm chỉ có một nút (`T[g] = 1`) thì mọi `c_g(T − c_g) = 0` — nó không đóng góp.
- Nếu `c_g(v) = T[g]` (cả nhóm nằm trong cây con) thì cũng đóng góp 0 cho cạnh đó.

**Bất biến.** Sau khi xử lý xong mọi con của `v` (và chính `v`), `cnt` là số đếm chính xác theo nhóm của cây con `v`, `A = Σ c_g·T[g]`, `B = Σ c_g²`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đếm theo cạnh với `dict` **sao chép** (`O(n²)` trường hợp xấu)

Công thức 6.1, nhưng gộp bằng cách **tạo bản sao mới** của dict cho mỗi nút. Đúng nhưng chậm với cây dạng đường thẳng nhiều nhóm: mỗi nút sao chép dict có tới `O(n)` phần tử.

In [8]:
def total_cost_copy_maps(n, edges, group):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    parent, order = bfs_order(n, adj)
    T = Counter(group)
    cnt = [None] * n                              # cnt[v]: dict của cây con v
    total = 0
    for v in reversed(order):
        mine = {group[v]: 1}
        for w in adj[v]:
            if w != parent[v]:
                for g, x in cnt[w].items():       # sao chép cả dict của con
                    mine[g] = mine.get(g, 0) + x
        cnt[v] = mine
        if v != 0:
            total += sum(c * (T[g] - c) for g, c in mine.items())
    return total

for n, edges, group, e in EXAMPLES:
    print(total_cost_copy_maps(n, edges, group), e)

4 4
2 2
3 3
0 0


**Độ phức tạp.** Mỗi nút quét toàn bộ dict của các con và dict của chính nó: `O(n²)` khi đường thẳng với `n` nhóm khác nhau (dict ở độ sâu `d` có `~d` phần tử). Đúng đắn nhưng quá chậm; dùng làm trọng tài thứ hai.

### 7.2. Cách 3 — Gộp nhỏ vào lớn + cập nhật `A`, `B` tăng dần (bản nộp)

**Giải thích từng đoạn.**

1. Dựng `adj`, BFS để có `parent`, `order`.
2. `T = Counter(group)`.
3. Với mỗi `v`: `cnt[v] = {group[v]: 1}`, `A[v] = T[group[v]]`, `B[v] = 1`.
4. Duyệt `order` ngược. Cộng `A[v] − B[v]` vào đáp án nếu `v ≠ 0`. Gộp vào cha `p`: hoán đổi nếu cần để dict `p` là dict lớn hơn; với mỗi `(g, x)` trong dict nhỏ cập nhật `A, B` và `cnt`.
5. Trả đáp án.

**Mẹo hoán đổi:** vì `cnt`, `A`, `B` là ba mảng, ta hoán đổi cả ba giá trị tại `v` và `p`; sau đó dict ở `p` là dict lớn, `v` giữ dict nhỏ để duyệt.

In [9]:
def total_cost_small_to_large(n, edges, group):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    parent, order = bfs_order(n, adj)
    T = Counter(group)
    cnt = [{group[v]: 1} for v in range(n)]
    A = [T[group[v]] for v in range(n)]
    B = [1] * n
    total = 0
    for v in reversed(order):
        if v != 0:
            total += A[v] - B[v]                  # đóng góp của cạnh (v, parent[v])
        p = parent[v]
        if p < 0:
            continue
        if len(cnt[v]) > len(cnt[p]):             # gộp nhỏ vào lớn
            cnt[v], cnt[p] = cnt[p], cnt[v]
            A[v], A[p] = A[p], A[v]
            B[v], B[p] = B[p], B[v]
        mp = cnt[p]
        a, b = A[p], B[p]
        for g, x in cnt[v].items():
            y = mp.get(g, 0)
            a += x * T[g]
            b += (y + x) * (y + x) - y * y
            mp[g] = y + x
        A[p], B[p] = a, b
        cnt[v] = None                             # giải phóng bộ nhớ
    return total

for n, edges, group, e in EXAMPLES:
    print(total_cost_small_to_large(n, edges, group), e)

4 4
2 2
3 3
0 0


**Độ phức tạp.** Mỗi lần ta duyệt dict **nhỏ**, mỗi phần tử `(g, x)` bị di chuyển; mỗi nhóm di chuyển nhiều nhất `log2(n)` lần (dict chứa nó ít nhất gấp đôi sau mỗi lần). Tổng `O(n log n)` phép dict (trung bình `O(1)` mỗi phép). Bộ nhớ `O(n)`.

### 7.3. Cách 4 — Cây ảo (virtual tree) theo từng nhóm + LCA

Khác hoàn toàn hướng "gộp bản đồ": xử lý **từng nhóm** độc lập. Với nhóm `g` có `k` nút, tổng khoảng cách mọi cặp bằng `Σ_cạnh trong cây Steiner len · c · (k − c)`, trong đó cạnh nằm trên **cây ảo** của nhóm: cây gồm các nút của nhóm cộng LCA của các cặp kề nhau theo thứ tự DFS. Mỗi cạnh ảo là một đường dài `len = dep[con] − dep[cha]` trên cây gốc, và mọi cạnh trên đường đó có cùng `c`.

**Các bước.**

1. DFS (không đệ quy) cho `tin`, `tout`, `dep`, `parent` và thứ tự tiền tự `pre`.
2. LCA bằng **bảng thưa** (sparse table) trên mảng `dep·n + nút` theo thứ tự tiền tự: với `tin[u] < tin[v]`, `LCA = parent[ nút có độ sâu nhỏ nhất trong pre[tin[u]+1 .. tin[v]] ]`.
3. Với mỗi nhóm: sắp theo `tin`, thêm LCA các cặp kề, sắp lại, dựng cha bằng ngăn xếp (dùng `tin/tout` kiểm tra tổ tiên), cộng dồn số đếm từ dưới lên và cộng `len · c · (k − c)`.

In [10]:
def total_cost_virtual_tree(n, edges, group):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    # DFS không đệ quy: tin theo thứ tự tiền tự, tout = tin lớn nhất trong cây con
    parent = [-1] * n; dep = [0] * n; tin = [0] * n; tout = [0] * n
    order = [0]; visited = [False] * n; visited[0] = True
    ptr = [0] * n
    stack = [0]
    while stack:
        v = stack[-1]
        if ptr[v] < len(adj[v]):
            w = adj[v][ptr[v]]; ptr[v] += 1
            if not visited[w]:
                visited[w] = True
                parent[w] = v; dep[w] = dep[v] + 1
                tin[w] = len(order); order.append(w); stack.append(w)
        else:
            tout[v] = len(order) - 1
            stack.pop()
    # Bảng thưa trên mảng khoá = dep * n + nút, theo thứ tự tiền tự
    vals = [dep[x] * n + x for x in order]
    sp = [vals]
    j = 1
    while (1 << j) <= n:
        prev, h = sp[-1], 1 << (j - 1)
        sp.append([min(prev[i], prev[i + h]) for i in range(n - (1 << j) + 1)])
        j += 1

    def lca(u, v):
        if u == v:
            return u
        a, b = tin[u], tin[v]
        if a > b:
            a, b = b, a
        a += 1
        k = (b - a + 1).bit_length() - 1
        val = min(sp[k][a], sp[k][b - (1 << k) + 1])
        return parent[val % n]

    by_group = {}
    for v in range(n):
        by_group.setdefault(group[v], []).append(v)
    total = 0
    for g, ns in by_group.items():
        k = len(ns)
        if k < 2:
            continue
        ns.sort(key=lambda x: tin[x])
        nodes = set(ns)
        for a, b in zip(ns, ns[1:]):
            nodes.add(lca(a, b))
        nodes = sorted(nodes, key=lambda x: tin[x])
        par, st = {}, []
        for v in nodes:
            while st and tout[st[-1]] < tin[v]:
                st.pop()
            par[v] = st[-1] if st else -1
            st.append(v)
        c = {v: 0 for v in nodes}
        for v in ns:
            c[v] = 1
        for v in reversed(nodes):
            p = par[v]
            if p >= 0:
                total += (dep[v] - dep[p]) * c[v] * (k - c[v])
                c[p] += c[v]
    return total

for n, edges, group, e in EXAMPLES:
    print(total_cost_virtual_tree(n, edges, group), e)

4 4
2 2
3 3
0 0


**Độ phức tạp.** Xây bảng thưa `O(n log n)`; mỗi nhóm `k` nút: sắp xếp `O(k log k)`, `k − 1` truy vấn LCA `O(1)`, dựng cây ảo `O(k)`. Tổng `O(n log n)` thời gian, `O(n log n)` bộ nhớ cho bảng thưa. Cách này hay khi cần xử lý nhóm riêng lẻ, nhưng code dài hơn Cách 3.

## 8. TRACE từng bước

### 8.1. Gộp nhỏ vào lớn trên đường `0–1–2–3–4` với `group = [1,2,1,2,1]`

`T[1] = 3`, `T[2] = 2`. Duyệt `order` ngược: `4, 3, 2, 1, 0`. Hàm trace in `(cnt, A, B)` của từng nút trước/sau khi gộp, đóng góp `A − B` của mỗi cạnh và tổng chạy.

In [11]:
def trace_small_to_large(n, edges, group):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    parent, order = bfs_order(n, adj)
    T = Counter(group)
    print("T =", dict(T), "| parent =", parent, "| order =", order)
    cnt = [{group[v]: 1} for v in range(n)]
    A = [T[group[v]] for v in range(n)]
    B = [1] * n
    total = 0
    for v in reversed(order):
        print(f"\n== xử lý nút {v}: cnt={cnt[v]}, A={A[v]}, B={B[v]}")
        if v != 0:
            total += A[v] - B[v]
            print(f"   cạnh ({v},{parent[v]}): đóng góp A−B = {A[v] - B[v]}, tổng chạy = {total}")
        p = parent[v]
        if p < 0:
            continue
        if len(cnt[v]) > len(cnt[p]):
            print(f"   dict của {v} (kích thước {len(cnt[v])}) lớn hơn dict của cha {p} ({len(cnt[p])}) -> hoán đổi")
            cnt[v], cnt[p] = cnt[p], cnt[v]; A[v], A[p] = A[p], A[v]; B[v], B[p] = B[p], B[v]
        mp = cnt[p]; a, b = A[p], B[p]
        for g, x in cnt[v].items():
            y = mp.get(g, 0)
            a += x * T[g]; b += (y + x) ** 2 - y * y; mp[g] = y + x
            print(f"   gộp nhóm {g}: y={y} + x={x} -> {y + x}; A += {x}*{T[g]}, B += {(y + x) ** 2 - y * y}")
        A[p], B[p] = a, b
        print(f"   cha {p}: cnt={mp}, A={a}, B={b}")
        cnt[v] = None
    print("\nTổng =", total)
    return total

trace_small_to_large(5, [[0, 1], [1, 2], [2, 3], [3, 4]], [1, 2, 1, 2, 1])

T = {1: 3, 2: 2} | parent = [-1, 0, 1, 2, 3] | order = [0, 1, 2, 3, 4]

== xử lý nút 4: cnt={1: 1}, A=3, B=1
   cạnh (4,3): đóng góp A−B = 2, tổng chạy = 2
   gộp nhóm 1: y=0 + x=1 -> 1; A += 1*3, B += 1
   cha 3: cnt={2: 1, 1: 1}, A=5, B=2

== xử lý nút 3: cnt={2: 1, 1: 1}, A=5, B=2
   cạnh (3,2): đóng góp A−B = 3, tổng chạy = 5
   dict của 3 (kích thước 2) lớn hơn dict của cha 2 (1) -> hoán đổi
   gộp nhóm 1: y=1 + x=1 -> 2; A += 1*3, B += 3
   cha 2: cnt={2: 1, 1: 2}, A=8, B=5

== xử lý nút 2: cnt={2: 1, 1: 2}, A=8, B=5
   cạnh (2,1): đóng góp A−B = 3, tổng chạy = 8
   dict của 2 (kích thước 2) lớn hơn dict của cha 1 (1) -> hoán đổi
   gộp nhóm 2: y=1 + x=1 -> 2; A += 1*2, B += 3
   cha 1: cnt={2: 2, 1: 2}, A=10, B=8

== xử lý nút 1: cnt={2: 2, 1: 2}, A=10, B=8
   cạnh (1,0): đóng góp A−B = 2, tổng chạy = 10
   dict của 1 (kích thước 2) lớn hơn dict của cha 0 (1) -> hoán đổi
   gộp nhóm 1: y=2 + x=1 -> 3; A += 1*3, B += 5
   cha 0: cnt={2: 2, 1: 3}, A=13, B=13

== xử lý nút 0: cnt={

10

Kiểm tra bằng tay: các cặp nhóm 1 `(0,2)=2, (0,4)=4, (2,4)=2` ⇒ 8; nhóm 2 `(1,3) = 2`. Tổng `10`.

### 8.2. Trace ví dụ 3 (sao) và ví dụ 2

Quan sát `E(v) = A − B` bằng `Σ c(T−c)` của cây con.

In [12]:
trace_small_to_large(*EXAMPLES[2][:3])
trace_small_to_large(*EXAMPLES[1][:3])

T = {1: 2, 4: 2} | parent = [-1, 0, 0, 0] | order = [0, 1, 2, 3]

== xử lý nút 3: cnt={4: 1}, A=2, B=1
   cạnh (3,0): đóng góp A−B = 1, tổng chạy = 1
   gộp nhóm 4: y=0 + x=1 -> 1; A += 1*2, B += 1
   cha 0: cnt={1: 1, 4: 1}, A=4, B=2

== xử lý nút 2: cnt={4: 1}, A=2, B=1
   cạnh (2,0): đóng góp A−B = 1, tổng chạy = 2
   gộp nhóm 4: y=1 + x=1 -> 2; A += 1*2, B += 3
   cha 0: cnt={1: 1, 4: 2}, A=6, B=5

== xử lý nút 1: cnt={1: 1}, A=2, B=1
   cạnh (1,0): đóng góp A−B = 1, tổng chạy = 3
   gộp nhóm 1: y=1 + x=1 -> 2; A += 1*2, B += 3
   cha 0: cnt={1: 2, 4: 2}, A=8, B=8

== xử lý nút 0: cnt={1: 2, 4: 2}, A=8, B=8

Tổng = 3
T = {3: 2, 2: 1} | parent = [-1, 0, 1] | order = [0, 1, 2]

== xử lý nút 2: cnt={3: 1}, A=2, B=1
   cạnh (2,1): đóng góp A−B = 1, tổng chạy = 1
   gộp nhóm 3: y=0 + x=1 -> 1; A += 1*2, B += 1
   cha 1: cnt={2: 1, 3: 1}, A=3, B=2

== xử lý nút 1: cnt={2: 1, 3: 1}, A=3, B=2
   cạnh (1,0): đóng góp A−B = 1, tổng chạy = 2
   dict của 1 (kích thước 2) lớn hơn dict của cha 0

2

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3. Tên phương thức `interactionCosts` là suy luận (xem ghi chú đầu notebook).

In [13]:
from collections import Counter


class Solution:
    def interactionCosts(self, n: int, edges: list[list[int]], group: list[int]) -> int:
        adj = [[] for _ in range(n)]
        for u, v in edges:
            adj[u].append(v)
            adj[v].append(u)

        # BFS từ gốc 0: parent và thứ tự duyệt (cha luôn đứng trước con)
        parent = [-1] * n
        order = [0]
        seen = [False] * n
        seen[0] = True
        for v in order:
            for w in adj[v]:
                if not seen[w]:
                    seen[w] = True
                    parent[w] = v
                    order.append(w)

        T = Counter(group)                       # T[g]: số nút nhóm g toàn cây
        cnt = [{group[v]: 1} for v in range(n)]  # cnt[v]: đếm theo nhóm trong cây con v
        A = [T[group[v]] for v in range(n)]      # A = sum_g c_g * T[g]
        B = [1] * n                              # B = sum_g c_g ** 2

        ans = 0
        for v in reversed(order):                # con trước cha
            if v:
                ans += A[v] - B[v]               # = sum_g c_g * (T[g] - c_g): đóng góp của cạnh (v, parent[v])
            p = parent[v]
            if p < 0:
                continue
            if len(cnt[v]) > len(cnt[p]):        # gộp nhỏ vào lớn
                cnt[v], cnt[p] = cnt[p], cnt[v]
                A[v], A[p] = A[p], A[v]
                B[v], B[p] = B[p], B[v]
            mp = cnt[p]
            a, b = A[p], B[p]
            for g, x in cnt[v].items():
                y = mp.get(g, 0)
                a += x * T[g]
                b += (y + x) * (y + x) - y * y
                mp[g] = y + x
            A[p], B[p] = a, b
            cnt[v] = None
        return ans


sol = Solution()
for n, edges, group, e in EXAMPLES:
    assert sol.interactionCosts(n, edges, group) == e, (n, edges, group)
print("4 ví dụ của đề: OK")

4 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**(a) Công thức cạnh.** Với hai nút `x, y` bất kỳ, đường đi giữa chúng là duy nhất; một cạnh nằm trên đường đó ⇔ `x, y` nằm ở hai phía khác nhau của cạnh. Do đó `dist(x, y) = #cạnh tách x khỏi y`, và tổng chi phí `= Σ_cạnh #cặp cùng nhóm bị tách`. Với cạnh `(v, parent[v])` và nhóm `g`: số cặp nhóm `g` bị tách là `c_g(v) · (T[g] − c_g(v))` (một nút ở dưới, một nút ở trên). Cộng trên mọi nhóm cho `E(v)`.

**(b) `A − B`.** `Σ c(T − c) = Σ cT − Σ c²` là đại số thuần tuý. Khi `c_g` đổi từ `y` sang `y + x`, `Σ cT` đổi `x · T[g]` và `Σ c²` đổi `(y + x)² − y²` — đúng như code.

**(c) Thứ tự xử lý.** BFS đặt cha trước con, nên duyệt ngược xử lý mọi con của `v` trước `v`; khi tới `v`, `cnt[v]` đã chứa số đếm của **toàn bộ** cây con của `v` (các con đã gộp vào `v` ở các bước trước; chính `v` được khởi tạo từ đầu).

**(d) Hoán đổi giữ đúng.** Hoán đổi `(cnt, A, B)` giữa `v` và `p` trước khi gộp tương đương việc gộp `p` vào `v` rồi đổi tên; kết quả `(cnt, A, B)` ở `p` luôn là bộ ba đúng của cây con `p` sau khi gộp xong `v`. Bộ ba cũ của `p` không còn dùng ở bước sau (ta chỉ đọc `cnt[v]`, đã gộp, rồi bỏ).

**(e) Độ phức tạp nhờ gộp nhỏ vào lớn.** Mỗi khoá `(nhóm)` bị duyệt khi nó nằm trong dict **nhỏ hơn**; sau khi gộp, dict chứa nó có kích thước ít nhất gấp đôi dict nhỏ cũ (xét theo số khoá). Vậy mỗi khoá được duyệt `≤ log2(n)` lần ⇒ `O(n log n)`.

## 11. Phân tích Time/Space

Đặt `n` là số nút.

| Cách | Time | Space |
|---|---|---|
| 1. BFS từ mỗi nút | `O(n²)` | `O(n)` |
| 2. Đếm theo cạnh, sao chép dict | `O(n²)` xấu nhất | `O(n²)` xấu nhất |
| 3. Gộp nhỏ vào lớn | `O(n log n)` | `O(n)` |
| 4. Cây ảo + LCA | `O(n log n)` | `O(n log n)` |

Ta đo trên `n = 10^5` với ba dạng cây: ngẫu nhiên, đường thẳng, sao.

In [14]:
import time
rng = random.Random(4018)
n = 100000
tests = {}
tests["ngẫu nhiên, 50 nhóm"] = ([[rng.randint(max(0, v - 5), v - 1), v] for v in range(1, n)], [rng.randint(1, 50) for _ in range(n)])
tests["đường thẳng, nhóm ngẫu nhiên"] = ([[v - 1, v] for v in range(1, n)], [rng.randint(1, n) for _ in range(n)])
tests["đường thẳng, 2 nhóm"] = ([[v - 1, v] for v in range(1, n)], [1 + v % 2 for v in range(n)])
tests["sao, mọi nhóm riêng"] = ([[0, v] for v in range(1, n)], [v + 1 for v in range(n)])
tests["cây nhị phân đầy đủ, 1000 nhóm"] = ([[(v - 1) // 2, v] for v in range(1, n)], [rng.randint(1, 1000) for _ in range(n)])
for name, (edges, group) in tests.items():
    t = time.time(); r1 = Solution().interactionCosts(n, edges, group); t1 = time.time() - t
    t = time.time(); r2 = total_cost_virtual_tree(n, edges, group); t2 = time.time() - t
    assert r1 == r2
    print(f"{name:34s} -> {r1}  | gộp nhỏ vào lớn {t1:.2f}s | cây ảo {t2:.2f}s")

ngẫu nhiên, 50 nhóm                -> 1109215690945  | gộp nhỏ vào lớn 0.18s | cây ảo 0.41s


đường thẳng, nhóm ngẫu nhiên       -> 1661145100  | gộp nhỏ vào lớn 0.19s | cây ảo 0.30s


đường thẳng, 2 nhóm                -> 83333333300000  | gộp nhỏ vào lớn 0.15s | cây ảo 0.20s


sao, mọi nhóm riêng                -> 0  | gộp nhỏ vào lớn 0.15s | cây ảo 0.25s


cây nhị phân đầy đủ, 1000 nhóm     -> 135845260  | gộp nhỏ vào lớn 0.21s | cây ảo 0.34s


## 12. Bộ test

Ví dụ đề, edge case, rồi đối chiếu 4 cách (và `Solution`) với BFS vét cạn trên cây ngẫu nhiên có seed — gồm cây ngẫu nhiên, đường thẳng, sao, và **gốc 0 nằm ở vị trí bất kỳ** (nhãn nút xáo trộn).

In [15]:
ALL = [total_cost_copy_maps, total_cost_small_to_large, total_cost_virtual_tree, Solution().interactionCosts]

EDGE = [
    (1, [], [1], 0),                                    # một nút
    (2, [[1, 0]], [5, 5], 1),                           # cạnh viết ngược
    (2, [[0, 1]], [2, 2], 1),
    (5, [[0, 1], [0, 2], [0, 3], [0, 4]], [1, 2, 2, 2, 1], 7),   # sao: nhóm 1 gồm (0,4)=1; nhóm 2 ba nút = 3 cặp x 2
    (4, [[0, 1], [1, 2], [2, 3]], [7, 7, 7, 7], 10),             # đường: 1+1+1+2+2+3
    (4, [[0, 1], [1, 2], [2, 3]], [1, 2, 3, 4], 0),              # mọi nhóm khác nhau
]
for n, edges, group, e in EXAMPLES + EDGE:
    assert total_cost_bruteforce(n, edges, group) == e, ("trọng tài", n, edges, group)
    for fn in ALL:
        assert fn(n, edges, group) == e, (fn.__name__, n, edges, group, fn(n, edges, group), e)
print("Ví dụ đề + edge case: mọi cách đúng")


def random_tree(rng, n, kind):
    edges = []
    for v in range(1, n):
        if kind == "path":
            u = v - 1
        elif kind == "star":
            u = 0
        elif kind == "deep":
            u = rng.randint(max(0, v - 3), v - 1)
        else:
            u = rng.randint(0, v - 1)
        edges.append([u, v] if rng.random() < 0.5 else [v, u])
    perm = list(range(n)); rng.shuffle(perm)               # gốc 0 ở vị trí tuỳ ý
    return [[perm[a], perm[b]] for a, b in edges]


rng = random.Random(401801)
for _ in range(600):
    n = rng.randint(1, 30)
    kind = rng.choice(["random", "path", "star", "deep"])
    edges = random_tree(rng, n, kind)
    group = [rng.randint(1, rng.choice([1, 2, 3, n])) for _ in range(n)]
    ref = total_cost_bruteforce(n, edges, group)
    for fn in ALL:
        assert fn(n, edges, group) == ref, (fn.__name__, n, edges, group, fn(n, edges, group), ref)
print("600 cây ngẫu nhiên n<=30: mọi cách khớp BFS vét cạn")

Ví dụ đề + edge case: mọi cách đúng
600 cây ngẫu nhiên n<=30: mọi cách khớp BFS vét cạn


In [16]:
rng = random.Random(7)
for _ in range(60):
    n = rng.randint(100, 400)
    edges = random_tree(rng, n, rng.choice(["random", "path", "deep"]))
    group = [rng.randint(1, rng.choice([2, 10, n])) for _ in range(n)]
    ref = total_cost_bruteforce(n, edges, group)
    assert Solution().interactionCosts(n, edges, group) == ref == total_cost_virtual_tree(n, edges, group)
print("60 cây 100..400 nút: Solution = cây ảo = BFS vét cạn")

# Input không bị sửa
edges = [[0, 1], [1, 2]]; group = [1, 1, 1]
e0, g0 = [r[:] for r in edges], group[:]
Solution().interactionCosts(3, edges, group)
assert edges == e0 and group == g0
print("edges và group không bị thay đổi")

60 cây 100..400 nút: Solution = cây ảo = BFS vét cạn
edges và group không bị thay đổi


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng đệ quy cho cây sâu

**Code sai:** `def dfs(v, p): ... dfs(w, v)` cho cây đường thẳng `10^5` nút.
**Vì sao sai:** `RecursionError` (giới hạn mặc định ~1000), dù tăng `setrecursionlimit` cũng dễ tràn ngăn xếp C.
**Code đúng:** BFS lấy `order`, rồi duyệt ngược.

In [17]:
import sys
def dfs_sum_sizes(n, adj):
    def dfs(v, p):
        total = 1
        for w in adj[v]:
            if w != p:
                total += dfs(w, v)
        return total
    return dfs(0, -1)

n = 5000
adj = [[] for _ in range(n)]
for v in range(1, n):
    adj[v - 1].append(v); adj[v].append(v - 1)
try:
    dfs_sum_sizes(n, adj)
except RecursionError as ex:
    print("RecursionError với n = 5000:", type(ex).__name__)

RecursionError với n = 5000: RecursionError


### Lỗi 2 — Quên chia/đếm trùng cặp: dùng `c_g · T[g]` thay vì `c_g · (T[g] − c_g)`

**Vì sao sai:** `c · T` đếm cả cặp nằm cùng phía (không đi qua cạnh) và cặp có thứ tự hai lần.
**Code đúng:** `c * (T - c)`.

In [18]:
def wrong_edge_formula(n, edges, group):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    parent, order = bfs_order(n, adj)
    T = Counter(group)
    cnt = [None] * n
    total = 0
    for v in reversed(order):
        mine = {group[v]: 1}
        for w in adj[v]:
            if w != parent[v]:
                for g, x in cnt[w].items():
                    mine[g] = mine.get(g, 0) + x
        cnt[v] = mine
        if v:
            total += sum(c * T[g] for g, c in mine.items())     # SAI
    return total

for n, edges, group, e in EXAMPLES[:3]:
    print("sai:", wrong_edge_formula(n, edges, group), "| đúng:", e)

sai: 9 | đúng: 4
sai: 5 | đúng: 2
sai: 6 | đúng: 3


### Lỗi 3 — Luôn đổ dict của con vào dict của cha (không hoán đổi) → `O(n²)`

**Code sai:** `for g, x in cnt[v].items(): cnt[p][g] += x` mà không kiểm tra kích thước.
**Vì sao sai (về tốc độ):** trên đường thẳng với nhiều nhóm khác nhau, nút `v` ở độ sâu `d` có dict `~n − d` phần tử còn cha `p` chỉ có 1 phần tử — ta duyệt phần tử của dict **lớn** ở mỗi bước ⇒ `O(n²)`.
**Code đúng:** `if len(cnt[v]) > len(cnt[p]): hoán đổi`, rồi mới duyệt dict nhỏ.

In [19]:
def count_moves(n, small_to_large):
    # Đường thẳng 0-1-...-(n-1), mỗi nút một nhóm khác nhau; đếm số phần tử bị duyệt khi gộp
    cnt = [{v: 1} for v in range(n)]
    moves = 0
    for v in range(n - 1, 0, -1):          # gộp nút v vào cha v-1
        p = v - 1
        if small_to_large and len(cnt[v]) > len(cnt[p]):
            cnt[v], cnt[p] = cnt[p], cnt[v]
        for g, x in cnt[v].items():
            cnt[p][g] = cnt[p].get(g, 0) + x
            moves += 1
        cnt[v] = None
    return moves

for n in (500, 1000, 2000):
    print(n, "| không hoán đổi:", count_moves(n, False), "| gộp nhỏ vào lớn:", count_moves(n, True))

500 | không hoán đổi: 124750 | gộp nhỏ vào lớn: 499
1000 | không hoán đổi: 499500 | gộp nhỏ vào lớn: 999
2000 | không hoán đổi: 1999000 | gộp nhỏ vào lớn: 1999


### Lỗi 4 — Cộng đóng góp của gốc

**Code sai:** cộng `A[0] − B[0]`.
**Vì sao sai:** gốc không có cạnh lên cha; thực ra `A[0] − B[0] = Σ T·T − Σ T² = 0` nên vô hại, nhưng nếu hoán đổi làm lẫn lộn thì dễ cộng nhầm. **Code đúng:** chỉ cộng khi `v != 0`.

### Lỗi 5 — Giả sử cạnh `[u, v]` luôn có `u` là cha

**Vì sao sai:** đề không bảo đảm hướng. **Code đúng:** thêm cả hai chiều vào `adj`, rồi BFS từ 0 để xác định cha.

## 14. Mẹo & nhận dạng pattern

- **"Tổng khoảng cách trên cây" ** → **đóng góp của cạnh**: `Σ_cạnh (phía này) × (phía kia)`. Với điều kiện cùng nhóm thì nhân theo từng nhóm.
- **"Thông tin theo nhãn của cây con, nhãn nhiều"** → dict + **gộp nhỏ vào lớn** (DSU on tree) `O(n log n)`.
- Duy trì **tổng phụ thuộc đếm** (như `Σ c²`) bằng cập nhật tăng dần theo từng thay đổi nhỏ thay vì tính lại.
- Duyệt cây bằng **BFS + `order` ngược** tránh đệ quy.
- Khi nhóm là "ít nhóm lớn", có thể tính từng nhóm bằng DFS riêng (`O(n · số nhóm)`); khi nhiều nhóm → hướng cây ảo / gộp nhỏ vào lớn.
- Đối chiếu bằng BFS từ mỗi nút trên cây nhỏ; thử cây đường thẳng, sao, ngẫu nhiên và **xáo trộn nhãn** để gốc 0 không đặc biệt.

## 15. Biến thể

1. **Tổng khoảng cách mọi cặp** (không cần cùng nhóm) — chỉ số Wiener: `Σ_cạnh size · (n − size)` (Bài 5).
2. **Tổng trọng số cạnh:** nhân thêm `w_cạnh` cho mỗi đóng góp.
3. **Nhóm ≤ 20** (phiên bản I của bài): với mỗi nhóm một DFS đếm kích thước, `O(20 · n)`.
4. **Khoảng cách trung bình** hoặc **đường kính** theo nhóm: dùng cây ảo và tính trên cây Steiner.
5. **Truy vấn cập nhật nhãn:** cần cấu trúc động (HLD + cấu trúc đếm), khó hơn nhiều.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS từ mỗi nút | Nguyên văn | `O(n²)` | `O(n)` | Trọng tài, `n ≤ 400` |
| 2. Đếm theo cạnh, sao chép dict | `Σ c (T − c)` mỗi cạnh | `O(n²)` xấu | `O(n²)` xấu | Học công thức |
| 3. Gộp nhỏ vào lớn + `A, B` | Cập nhật tăng dần | `O(n log n)` | `O(n)` | **Bản nộp** |
| 4. Cây ảo + LCA | Từng nhóm một | `O(n log n)` | `O(n log n)` | Khi cần xử lý từng nhóm |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `n = 4`, `edges = [[0,1],[1,2],[2,3]]`, `group = [7,7,7,7]`. Tổng chi phí bằng bao nhiêu?

**Bài 2 (tính tay).** Sao 0 với các lá `1, 2, 3, 4`, `group = [1, 2, 2, 2, 1]`. Tính đóng góp từng cạnh bằng `Σ c (T − c)` và tổng.

**Bài 3 (điền TODO).** Hoàn thiện `edge_contribution(cnt, T)` trả về `Σ_g c_g · (T[g] − c_g)` cho dict `cnt` của một cây con.

**Bài 4 (tự viết lại).** Viết `my_total_cost(n, edges, group)` theo Cách 2 (đếm theo cạnh bằng dict của từng cây con, dùng BFS + `order` ngược).

**Bài 5 (mini challenge).** Viết `wiener_index(n, edges)`: tổng khoảng cách **mọi** cặp nút (không xét nhóm) trong `O(n)`; đối chiếu BFS.

In [20]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def edge_contribution(cnt, T):
    # TODO: tổng c * (T[g] - c) cho mọi (g, c) trong cnt.items()
    pass

# Bài 4 — TODO
def my_total_cost(n, edges, group):
    # TODO: adj, bfs_order, T = Counter(group); duyệt order ngược; gộp dict con; cộng edge_contribution khi v != 0
    pass

# Bài 5 — TODO
def wiener_index(n, edges):
    # TODO: sz[v] = kích thước cây con; tổng sz[v] * (n - sz[v]) cho v != 0
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: 6 cặp.
- Mức 2: khoảng cách `1,1,1,2,2,3`.
- Mức 3: tổng `10`. Hoặc theo cạnh: `1·3 + 2·2 + 3·1 = 10`.

**Bài 2.**
- Mức 1: `T[1] = 2`, `T[2] = 3`.
- Mức 2: lá `1, 2, 3` thuộc nhóm 2: mỗi lá `c = 1`, đóng góp `1·(3−1) = 2`. Lá 4 thuộc nhóm 1: `1·(2−1) = 1`.
- Mức 3: `2 + 2 + 2 + 1 = 7`.

**Bài 3.**
- Mức 1: duyệt `cnt.items()`.
- Mức 2: `c * (T[g] - c)`.
- Mức 3: `sum(c * (T[g] - c) for g, c in cnt.items())`.

**Bài 4.**
- Mức 1: giống Cách 2 ở mục 7.1.
- Mức 2: `cnt[v] = {group[v]: 1}` rồi gộp dict của các con.
- Mức 3: cộng `edge_contribution(cnt[v], T)` khi `v != 0`.

**Bài 5.**
- Mức 1: mỗi cạnh chia `n` nút thành `s` và `n − s`.
- Mức 2: `s` là kích thước cây con.
- Mức 3: duyệt `order` ngược cộng kích thước vào cha.

## 19. Đáp án tham khảo

In [21]:
# Bài 1, 2
assert total_cost_bruteforce(4, [[0, 1], [1, 2], [2, 3]], [7, 7, 7, 7]) == 10
assert Solution().interactionCosts(5, [[0, 1], [0, 2], [0, 3], [0, 4]], [1, 2, 2, 2, 1]) == 7

# Bài 3
def edge_contribution_ans(cnt, T):
    return sum(c * (T[g] - c) for g, c in cnt.items())
assert edge_contribution_ans({1: 1}, Counter([1, 1, 4, 4])) == 1
assert edge_contribution_ans({2: 3, 1: 1}, Counter([1, 2, 2, 2, 1])) == 3 * 0 + 1 * 1
print("Bài 1–3 OK")

Bài 1–3 OK


In [22]:
# Bài 4
def my_total_cost_ans(n, edges, group):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    parent, order = bfs_order(n, adj)
    T = Counter(group)
    cnt = [None] * n
    total = 0
    for v in reversed(order):
        mine = {group[v]: 1}
        for w in adj[v]:
            if w != parent[v]:
                for g, x in cnt[w].items():
                    mine[g] = mine.get(g, 0) + x
        cnt[v] = mine
        if v != 0:
            total += edge_contribution_ans(mine, T)
    return total

# Bài 5
def wiener_index_ans(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    parent, order = bfs_order(n, adj)
    sz = [1] * n
    total = 0
    for v in reversed(order):
        if v != 0:
            total += sz[v] * (n - sz[v])
            sz[parent[v]] += sz[v]
    return total

rng = random.Random(99)
for _ in range(300):
    n = rng.randint(1, 20)
    edges = random_tree(rng, n, rng.choice(["random", "path", "star", "deep"]))
    group = [rng.randint(1, 3) for _ in range(n)]
    assert my_total_cost_ans(n, edges, group) == Solution().interactionCosts(n, edges, group)
    assert wiener_index_ans(n, edges) == total_cost_bruteforce(n, edges, [1] * n)     # mọi nút cùng một nhóm
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
parent, order = BFS từ 0          # cha trước con
T = Counter(group)
cnt[v] = {group[v]: 1};  A[v] = T[group[v]];  B[v] = 1
for v in reversed(order):         # con trước cha
    if v: ans += A[v] - B[v]      # = sum_g c_g * (T_g - c_g)
    p = parent[v]
    if len(cnt[v]) > len(cnt[p]): hoán đổi (cnt, A, B) của v và p
    for g, x in cnt[v].items():   # đổ dict nhỏ vào dict lớn
        y = cnt[p].get(g, 0)
        A[p] += x * T[g];  B[p] += (y + x)**2 - y*y;  cnt[p][g] = y + x
```

**Checklist trước khi submit:**

- [ ] Đóng góp của cạnh là `Σ c (T − c)`, không phải `Σ c · T`.
- [ ] Duyệt bằng BFS + `order` ngược, không đệ quy.
- [ ] Gộp **nhỏ vào lớn**: hoán đổi cả `cnt`, `A`, `B`.
- [ ] Không cộng đóng góp của gốc 0.
- [ ] Thêm cạnh vô hướng vào cả hai chiều.
- [ ] Số nguyên lớn (đến `~10^15`) — Python không tràn.
- [ ] Đối chiếu BFS vét cạn trên cây nhỏ có seed (cả đường thẳng, sao, gốc bất kỳ).